# 接口自动化测试

学习目标：为小型 API 编写可重复的成功与失败测试，隔离测试数据，可靠恢复依赖替换，并选择合适的同步或异步测试客户端。

前置知识：pytest、断言、测试隔离、HTTP 请求、依赖注入、协程与上下文管理器。

适用版本：Python 3.12；异步 pytest 示例使用 AnyIO 的 asyncio 后端。完整依赖版本见环境入口。

环境准备：[FastAPI 环境与运行入口](README.md)。工作目录为 content/Web与应用开发/FastAPI；从空内核自上而下运行。本章先在 Notebook 中逐个观察测试，再用一个小型测试文件运行 pytest；不启动监听端口。

配套脚本：位于 scripts/07-api-testing/。

1. [test_api.py](scripts/07-api-testing/test_api.py)：集中展示本章的参数化和异步测试，供 pytest 从文件收集；应用只有查询与创建两条路由。

## 1 先检查一次成功响应

接口测试检查调用方看到的行为。先为记录查询编写两个断言：状态码应为 200，JSON 内容应包含约定的记录，而不只是“请求没有报错”。

TestClient 在应用内调用 FastAPI，不经过真实监听端口。测试代码使用普通 def 和同步调用；即使被测路由使用 async def，也不需要把这类测试改为协程。

In [1]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient


app = FastAPI()


@app.get("/records/{record_id}")
def read_record(record_id: int):
    if record_id != 1:
        raise HTTPException(status_code=404, detail="记录不存在")
    return {"id": 1, "title": "阅读文档"}


def test_read_record():
    with TestClient(app) as client:
        response = client.get("/records/1")
    assert response.status_code == 200
    assert response.json() == {"id": 1, "title": "阅读文档"}


test_read_record()
print("成功响应的状态码与内容符合约定")  # 预期：成功响应的状态码与内容符合约定。

C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


成功响应的状态码与内容符合约定


## 2 分开检查业务失败与请求校验失败

测试应覆盖接口主动拒绝的输入。这里，整数编号 99 可以进入查询逻辑，但没有对应记录，返回 404；非整数编号无法通过路径参数校验，返回 422。

业务错误可以检查约定的完整 JSON；框架校验错误则检查状态码、参数位置和错误类别，避免让测试依赖整段提示文字。

In [2]:
with TestClient(app) as client:
    missing = client.get("/records/99")
    invalid = client.get("/records/word")

assert missing.status_code == 404
assert missing.json() == {"detail": "记录不存在"}
assert invalid.status_code == 422
issue = invalid.json()["detail"][0]
assert issue["loc"] == ["path", "record_id"]
assert issue["type"] == "int_parsing"
print(missing.status_code, missing.json())  # 预期：404 {'detail': '记录不存在'}。
print(invalid.status_code, issue["loc"], issue["type"])  # 预期：422 ['path', 'record_id'] int_parsing。
# 这些 4xx 响应是预期业务结果，不是 pytest 测试失败。

404 {'detail': '记录不存在'}
422 ['path', 'record_id'] int_parsing


## 3 每次测试准备自己的数据

会修改数据的测试不能依赖另一个测试留下的记录。下面把应用和一个小字典一起放进 create_record_app：每次调用都创建独立字典，初始状态固定。

为观察数据是否串用，增加一个接收 title 查询参数的创建接口。本例只演示测试隔离，数据位于内存中。

In [3]:
def create_record_app() -> FastAPI:
    record_app = FastAPI()
    titles = {1: "阅读文档"}

    @record_app.get("/records/{record_id}")
    def read_record(record_id: int):
        if record_id not in titles:
            raise HTTPException(status_code=404, detail="记录不存在")
        return {"id": record_id, "title": titles[record_id]}

    @record_app.post("/records", status_code=201)
    def create_record(title: str):
        record_id = max(titles) + 1
        titles[record_id] = title
        return {"id": record_id, "title": title}

    return record_app

用一个测试同时观察写入是否成功、另一个独立应用是否仍保持初始状态。直接重复调用测试函数，也不应受到上一次写入的影响。

关闭客户端不会自动还原应用中的字典；隔离来自应用和数据的重新创建。数据库或文件测试也需要各自适用的数据准备与清理方式。

In [4]:
def test_data_isolation():
    with TestClient(create_record_app()) as first:
        created = first.post("/records", params={"title": "练习测试"})
        loaded = first.get("/records/2")
        assert created.status_code == 201
        assert loaded.status_code == 200
        assert loaded.json() == {"id": 2, "title": "练习测试"}

    with TestClient(create_record_app()) as second:
        assert second.get("/records/2").status_code == 404


test_data_isolation()
test_data_isolation()
print("重复两次：新记录都只出现在各自的应用中")  # 预期：重复两次：新记录都只出现在各自的应用中。

重复两次：新记录都只出现在各自的应用中

## 4 用参数化复用断言

pytest.mark.parametrize 让同一个测试函数使用多组参数运行。下面的 path 是请求路径，status 是预期状态码，expected 是预期 JSON；ids 为两组用例提供容易辨认的名称。

参数化不会自动复制可变参数。本例只读取 expected；每组用例都创建独立应用，避免依赖执行顺序。末节的配套测试文件保留这两组参数，让 pytest 实际收集运行。

In [5]:
import pytest


@pytest.mark.parametrize(
    "path,status,expected",
    [
        ("/records/1", 200, {"id": 1, "title": "阅读文档"}),
        ("/records/99", 404, {"detail": "记录不存在"}),
    ],
    ids=["found", "missing"],
)
def test_known_responses(path, status, expected):
    with TestClient(create_record_app()) as client:
        response = client.get(path)
    assert response.status_code == status
    assert response.json() == expected


# 普通函数调用只检查显式传入的一组；两组参数由 pytest 分别收集。
test_known_responses("/records/1", 200, {"id": 1, "title": "阅读文档"})
print("直接调用一组成功，参数化用例将在 pytest 中运行")  # 预期：直接调用一组成功，参数化用例将在 pytest 中运行。

直接调用一组成功，参数化用例将在 pytest 中运行


对断言结构不同的情况，可以保留独立测试。下面把前面的 422 检查整理为 test_ 开头的函数，可以直接调用；将同样的定义放进测试文件后，pytest 也能发现它。

In [6]:
def test_invalid_id():
    with TestClient(create_record_app()) as client:
        response = client.get("/records/word")
    assert response.status_code == 422
    issue = response.json()["detail"][0]
    assert issue["loc"] == ["path", "record_id"]
    assert issue["type"] == "int_parsing"


test_invalid_id()
print("非法编号指向 path 中的 record_id")  # 预期：非法编号指向 path 中的 record_id。

非法编号指向 path 中的 record_id


## 5 替换依赖后必须恢复

dependency_overrides 用“原依赖函数 → 替代函数”的映射替换依赖调用。测试可以提供固定输入，而不运行原依赖及其子依赖；替代函数的返回值会交给使用该依赖的路由。

先定义一个读取固定读者名称的小接口，再在测试中换成另一位读者。映射的键是函数对象，不是函数名称字符串。

In [7]:
from typing import Annotated

from fastapi import Depends


def current_reader() -> str:
    return "local-reader"


def create_reader_app() -> FastAPI:
    reader_app = FastAPI()

    @reader_app.get("/reader")
    def read_reader(name: Annotated[str, Depends(current_reader)]):
        return {"name": name}

    return reader_app

在 try 之前保存原有映射，在 finally 中恢复，避免测试中途失败后遗留替换。保存原有内容，也能保留此前已有的其他替换。

下面有意抛出一个明确的 RuntimeError 模拟测试中断，用 pytest.raises 核对该预期异常。请求断言失败等其他异常仍会传播；恢复后再次请求，确认使用原依赖。

In [8]:
def test_override_restores():
    reader_app = create_reader_app()
    # 先保留原映射，测试结束后恢复它，而不是直接清空全部依赖替换。
    previous = reader_app.dependency_overrides.copy()

    def test_reader() -> str:
        return "test-reader"

    with pytest.raises(RuntimeError, match="模拟测试中断"):
        try:
            reader_app.dependency_overrides[current_reader] = test_reader
            with TestClient(reader_app) as client:
                response = client.get("/reader")
                assert response.status_code == 200
                assert response.json() == {"name": "test-reader"}
            # 故意中断测试主体，观察 finally 仍然完成恢复。
            raise RuntimeError("模拟测试中断")
        finally:
            reader_app.dependency_overrides = previous

    # 用新客户端再次请求，确认恢复后的依赖已被使用。
    with TestClient(reader_app) as client:
        assert client.get("/reader").json() == {"name": "local-reader"}
    assert reader_app.dependency_overrides == previous


test_override_restores()
print("测试中断后恢复原依赖：local-reader")  # 预期：测试中断后恢复原依赖：local-reader。

测试中断后恢复原依赖：local-reader


## 6 用异步客户端在协程中测试

测试需要 await 异步操作时，可以使用 HTTPX AsyncClient。ASGITransport 将请求直接交给 ASGI 应用；这里的 base_url 用于组成请求地址，不会使请求连接到名为 test 的服务器。应用内测试通过，并不等于验证了 Uvicorn 启动和真实端口。

AnyIO 提供 pytest 的异步测试支持。pytest.mark.anyio 标记异步测试，anyio_backend 指定使用的后端；本例只运行 asyncio。Notebook 中先用 await 直接观察结果，pytest 中由插件运行协程。

In [9]:
from httpx import ASGITransport, AsyncClient


@pytest.mark.anyio
@pytest.mark.parametrize("anyio_backend", ["asyncio"])
async def test_async_lookup(anyio_backend):
    transport = ASGITransport(app=create_record_app())
    async with AsyncClient(
        transport=transport, base_url="http://test"
    ) as client:
        response = await client.get("/records/1")
    assert response.status_code == 200
    assert response.json() == {"id": 1, "title": "阅读文档"}


await test_async_lookup("asyncio")
print("异步请求得到同一条记录，客户端已退出上下文")  # 预期：异步请求得到同一条记录，客户端已退出上下文。

异步请求得到同一条记录，客户端已退出上下文


## 7 检查客户端是否执行 lifespan

有些应用在 lifespan 中准备资源。本节用 ready 标志代替真实资源，只观察测试是否执行已声明的启动和退出逻辑：进入时设为 True，退出时恢复 False。

TestClient 必须作为上下文管理器使用，才会运行 lifespan。HTTPX 的 ASGITransport 则不负责触发生命周期事件，进入 AsyncClient 上下文也不会自动启动应用。

In [10]:
from contextlib import asynccontextmanager


@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.ready = True
    try:
        yield
    finally:
        app.state.ready = False


def create_lifecycle_app() -> FastAPI:
    lifecycle_app = FastAPI(lifespan=lifespan)
    lifecycle_app.state.ready = False

    @lifecycle_app.get("/ready")
    def read_ready():
        return {"ready": lifecycle_app.state.ready}

    return lifecycle_app

先检查同步客户端的上下文边界。应用尚未进入生命周期、处理请求期间和关闭之后的状态分别可观察。

In [11]:
def test_client_lifespan():
    lifecycle_app = create_lifecycle_app()
    assert lifecycle_app.state.ready is False
    with TestClient(lifecycle_app) as client:
        assert client.get("/ready").json() == {"ready": True}
    assert lifecycle_app.state.ready is False


test_client_lifespan()
print("TestClient 上下文：未启动 → 已启动 → 已退出")  # 预期：TestClient 上下文：未启动 → 已启动 → 已退出。

TestClient 上下文：未启动 → 已启动 → 已退出

异步测试用 LifespanManager 明确管理启动和正常关闭，并将 manager.app 交给 ASGITransport；该包装保留生命周期状态的传递。管理器的启动和关闭默认各等待最多 5 秒。

下面先观察仅使用 AsyncClient 时 ready 仍为 False，再观察显式生命周期内为 True、正常退出后恢复 False。两个客户端都在自己的上下文退出时关闭。若异常从管理器内部冒出，asgi-lifespan 2.1.0 不执行正常 shutdown；测试通过的正常退出结果不能代表异常路径。

In [12]:
from asgi_lifespan import LifespanManager


@pytest.mark.anyio
@pytest.mark.parametrize("anyio_backend", ["asyncio"])
async def test_async_lifespan(anyio_backend):
    lifecycle_app = create_lifecycle_app()
    async with AsyncClient(
        transport=ASGITransport(app=lifecycle_app), base_url="http://test"
    ) as client:
        assert (await client.get("/ready")).json() == {"ready": False}

    async with LifespanManager(lifecycle_app) as manager:
        async with AsyncClient(
            transport=ASGITransport(app=manager.app), base_url="http://test"
        ) as client:
            assert (await client.get("/ready")).json() == {"ready": True}
    assert lifecycle_app.state.ready is False


await test_async_lifespan("asyncio")
print("仅 AsyncClient：未启动；显式 lifespan：已启动并完成退出")  # 预期：仅 AsyncClient：未启动；显式 lifespan：已启动并完成退出。

仅 AsyncClient：未启动；显式 lifespan：已启动并完成退出


## 8 用一个小测试文件运行 pytest

pytest 从测试文件收集用例。配套 test_api.py 集中保留本章的 create_record_app、参数化测试 test_known_responses 和异步测试 test_async_lookup；文件顶部列出需要的导入，不依赖当前 Notebook 内核。

打开文件可以直接读到完整代码：两组参数展开为两个用例，异步查询形成一个用例，共三个。前面其他测试已在 Notebook 中直接执行；它们不会被这个命令自动收集。要增加 pytest 用例，直接在文件里新增 test_ 开头的函数即可。

Step 1：在课程目录的终端运行配套测试文件。

```bash
python -B -m pytest -q -p no:cacheprovider scripts/07-api-testing/test_api.py
```

-q 显示简短摘要；-B 禁止写入 Python 字节码缓存，-p no:cacheprovider 关闭 pytest 的缓存插件，这两个选项让本次小实验不留下缓存文件。返回码为 0 表示收集到的测试全部通过。

也可以直接执行下面的 Code 单元，它在独立进程中运行同一条命令，并显示真实输出。sys.executable 指向当前内核所属的解释器；subprocess.run 的 timeout=60 为命令设置 60 秒上限。子进程退出后，测试模块不会留在 Notebook 的导入缓存中。

In [13]:
import os
import subprocess
import sys


result = subprocess.run(
    [sys.executable, "-B", "-m", "pytest", "-q", "-p", "no:cacheprovider",
     "scripts/07-api-testing/test_api.py"],
    capture_output=True, text=True, encoding="utf-8",
    env={**os.environ, "PYTHONUTF8": "1"}, timeout=60,
)
print(result.stdout, end="")  # 预期：pytest 汇总 3 passed；可能附依赖弃用警告，耗时与路径随环境变化。
print(result.stderr, end="")  # 预期：成功时通常为空；若工具向 stderr 写诊断，则原样显示。
result.check_returncode()  # 失败时保留报告并停止；当前文件共 3 个用例。

...                                                                      [100%]
============================== warnings summary ===============================
..\..\..\..\..\..\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1
  C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import TestClient as TestClient  # noqa

..\..\..\..\..\..\miniconda3\envs\hands-on-computing\Lib\site-packages\starlette\testclient.py:53
  C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\starlette\testclient.py:53: DeprecationWarning: The anyio.abc.BlockingPortal alias is deprecated, use anyio.from_thread.BlockingPortal instead.
    _PortalFactoryType = Callable[[], AbstractContextManager[anyio.abc.BlockingPortal]]

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnin

## 本章小结

（1）成功和失败响应都检查状态码与内容；预期的 404 或 422 并不代表测试失败。

（2）每个用例准备自己的可变数据，参数化用例也不依赖执行顺序；关闭客户端不等于还原数据。

（3）依赖替换的键是原函数对象，finally 负责恢复原有映射，包括测试提前中断的情况。

（4）同步 TestClient 和异步 ASGITransport 都属于应用内调用；选择异步客户端时还要明确由谁运行 lifespan。

## 练习

（1）在配套 test_api.py 的 test_known_responses 中增加编号 0 的用例，预期返回“记录不存在”。重新运行 pytest，确认收集到 4 个用例且全部通过。

（2）增加一个测试：先在一个应用中连续创建两条记录，确认得到编号 2、3；再创建独立应用，确认编号 2、3 均不存在。把测试函数写入配套 test_api.py，再运行 pytest，确认新用例被收集并通过。

（3）为读者依赖预先设置一个 baseline-reader 替换，再临时改为 test-reader。模拟测试中断后，确认恢复结果是 baseline-reader，原有替换没有被清空。

（4）增加一个异步测试：在 LifespanManager 内部用 pytest.raises 捕获受控的 ValueError，再请求 /ready，确认仍为 True；正常退出管理器后确认 ready 为 False。先在 Notebook 中 await 调用测试；需要交给 pytest 时，把生命周期定义、测试及其导入一同写入测试文件。

### 提示

（1）参数值列表与 ids 同步增加一项；每一组参数都是一个独立用例。

（2）数据隔离与测试执行顺序无关；先核对创建和读取成功，再核对另一应用的不存在响应。

（3）先保存带有原替换的字典，再进入 try/finally；仅调用 clear() 会丢掉原有配置。

（4）外层是 async with LifespanManager，内层才是 with pytest.raises；异常在管理器内部被核对和捕获，因此可以正常退出。保留异步测试的 AnyIO 标记与 asyncio 后端。

## 参考与引用来源

- FastAPI 官方文档：[Testing](https://fastapi.tiangolo.com/tutorial/testing/) 的 Using TestClient、Testing: extended example；[Path Parameters](https://fastapi.tiangolo.com/tutorial/path-params/#data-validation) 的 Data validation；[Testing Dependencies with Overrides](https://fastapi.tiangolo.com/advanced/testing-dependencies/) 的 dependency_overrides 映射与恢复；[Async Tests](https://fastapi.tiangolo.com/advanced/async-tests/) 的 AnyIO 标记、HTTPX 与生命周期提醒；[Testing Events](https://fastapi.tiangolo.com/advanced/testing-events/) 的 lifespan 上下文检查。支持同步与异步接口测试及依赖替换边界。
- pytest 官方文档：[Parametrize](https://docs.pytest.org/en/stable/how-to/parametrize.html#pytest-mark-parametrize-parametrizing-test-functions) 的多组参数、可变参数不自动复制；[Assertions](https://docs.pytest.org/en/stable/how-to/assert.html#assertions-about-expected-exceptions) 的预期异常；[Good Integration Practices](https://docs.pytest.org/en/stable/explanation/goodpractices.html#conventions-for-python-test-discovery) 的测试发现；[Exit codes](https://docs.pytest.org/en/stable/reference/exit-codes.html) 的返回码含义；[How to invoke pytest](https://docs.pytest.org/en/stable/how-to/usage.html) 的指定文件、python -m pytest 与 Disabling plugins。
- AnyIO 官方文档：[Testing with AnyIO](https://anyio.readthedocs.io/en/stable/testing.html) 的 Creating asynchronous tests、Specifying the backends to run on，支持 pytest.mark.anyio 与 anyio_backend 参数化。
- HTTPX 官方文档：[Transports](https://www.python-httpx.org/advanced/transports/#asgi-transport) 的 ASGI Transport、ASGI startup and shutdown；[Async Support](https://www.python-httpx.org/async/) 的客户端上下文管理，支持应用内异步请求与客户端关闭。
- Starlette 官方文档：[TestClient](https://starlette.dev/testclient/#testclient) 的上下文管理器与 lifespan 边界；[Lifespan](https://starlette.dev/lifespan/#running-lifespan-in-tests) 的测试内生命周期检查。
- GitHub 第一方项目：[asgi-lifespan](https://github.com/florimondmanca/asgi-lifespan#usage) 的 Usage、Accessing state 与 LifespanManager API，支持显式生命周期管理、manager.app 与默认等待上限。
- Python 3.12 官方文档：[命令行 -B](https://docs.python.org/3.12/using/cmdline.html#cmdoption-B)、[subprocess.run](https://docs.python.org/3.12/library/subprocess.html#subprocess.run)、[try 语句的 finally](https://docs.python.org/3.12/reference/compound_stmts.html#finally-clause)。支持字节码缓存控制、子进程执行与异常后的恢复。